In [1]:
import habitat_sim
import numpy as np
from PIL import Image
import os
from tqdm import tqdm
import pandas as pd

# Configuration
SCENE_FILE = "/Users/xinghebai/Desktop/Master Project/Sim Cross-view Dataset/replica_cad_baked_lighting/stages_uncompressed/Baked_sc3_staging_00.glb"

# Save images to a dedicated folder
OUTPUT_DIR_IMAGES = "sim_cross_view_dataset/images2"
os.makedirs(OUTPUT_DIR_IMAGES, exist_ok=True)

IMG_WIDTH = 256
IMG_HEIGHT = 256

In [2]:
# Setup the Simulator 
def setup_simulator(scene_path, width, height):
    sim_cfg = habitat_sim.SimulatorConfiguration()
    sim_cfg.scene_id = scene_path
    sim_cfg.enable_physics = False 

    camera_sensor_spec = habitat_sim.CameraSensorSpec()
    camera_sensor_spec.uuid = "rgb_sensor"
    camera_sensor_spec.sensor_type = habitat_sim.SensorType.COLOR
    camera_sensor_spec.resolution = [height, width]
    camera_sensor_spec.position = np.array([0, 1.5, 0], dtype=np.float32) 
    camera_sensor_spec.orientation = np.array([0, 0, 0], dtype=np.float32)
    
    agent_cfg = habitat_sim.AgentConfiguration()
    agent_cfg.sensor_specifications = [camera_sensor_spec]

    cfg = habitat_sim.Configuration(sim_cfg, [agent_cfg])
    sim = habitat_sim.Simulator(cfg)
    
    print(f"Simulator created for scene: {scene_path}")
    return sim


In [3]:
# Pose Sampling Logic
def get_camera_poses(sim):
    navmesh = sim.pathfinder
    if not navmesh.is_loaded:
        print("NavMesh not loaded, re-computing...")
        sim.recompute_navmesh(sim.pathfinder, habitat_sim.nav.NavMeshSettings())
        
    pose_samples = []

    num_points_to_sample = 600
    
    print(f"Sampling {num_points_to_sample} random camera poses with 4 yaw angles each...")
    
    for _ in range(num_points_to_sample): 
        spawn_point = navmesh.get_random_navigable_point()
        
        for yaw_degrees in [0, 90, 180, 270]:
            yaw_rad = np.deg2rad(yaw_degrees)
            Y_AXIS = np.array([0, 1, 0], dtype=np.float32)
            rotation = habitat_sim.utils.quat_from_angle_axis(
            yaw_rad, Y_AXIS
            )
            pose_samples.append((spawn_point, rotation, yaw_degrees))

    print(f"Generated {len(pose_samples)} camera poses.")
    return pose_samples

In [ ]:
def is_bad_image(image_array, brightness_threshold=35.0, variance_threshold=35.0):
    """
    Returns True if the image is likely invalid (too dark or no detail).
    
    Args:
        image_array: Numpy array of the image (H, W, C)
        brightness_threshold: Minimum average pixel intensity required.
        variance_threshold: Minimum standard deviation required (detects solid flat colors).
    """
    # Focus on RGB channels, ignore Alpha if present
    rgb_data = image_array[..., :3]
    
    # Check 1: Is it too dark? 
    mean_intensity = np.mean(rgb_data)
    if mean_intensity < brightness_threshold:
        return True
       
    # Check 2: Is it completely flat or solid color? 
    # If camera is clipped inside a solid grey floor object, variance will be near 0.
    std_dev = np.std(rgb_data)
    if std_dev < variance_threshold:
        return True
      
    return False

In [ ]:
# Main Generation Loop
def generate_images(sim, poses):
    agent = sim.initialize_agent(0)
    agent_state = habitat_sim.AgentState()
    scene_name = os.path.basename(sim.config.sim_cfg.scene_id)

    metadata_records = []
    
    print("Starting frame rendering...")
    
    saved_count = 0
    
    for i, (position, rotation, yaw) in enumerate(tqdm(poses, desc=f"Rendering {scene_name}")):
        agent_state.position = position
        agent_state.rotation = rotation
        agent.set_state(agent_state)
        
        obs = sim.get_sensor_observations()
        frame = obs["rgb_sensor"]

        # Filter check
        if is_bad_image(frame):
            continue # Skip this iteration, do not save
            
        frame_img = Image.fromarray(frame[..., :3], "RGB")

        # Save the image
        img_filename = f"{scene_name.replace('.', '_')}_frame_{saved_count:05d}.png"
        img_path_absolute = os.path.join(OUTPUT_DIR_IMAGES, img_filename)
        frame_img.save(img_path_absolute)

        metadata_records.append({
            "Filename": img_filename,
            "X": position[0],
            "Y": position[1], 
            "Z": position[2],
            "Yaw_Angle": yaw,
            "Score": None 
        })
        
        saved_count += 1
        
    if metadata_records:
        df = pd.DataFrame(metadata_records)
        excel_path = os.path.join("sim_cross_view_dataset", "image_data.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"Metadata saved to {excel_path}")

    print(f"Finished. Saved {saved_count} sequential images (skipped {len(poses) - saved_count} bad frames).")

In [6]:
if not os.path.exists(SCENE_FILE):
    print(f"Error: Scene file not found at {SCENE_FILE}")
else:
    simulator = setup_simulator(SCENE_FILE, IMG_WIDTH, IMG_HEIGHT)
    sampled_poses = get_camera_poses(simulator)
    generate_images(simulator, sampled_poses)
    simulator.close()
    print("Image generation complete.")

PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
[20:16:49:350616]:[Warning]:[Metadata] SceneDatasetAttributes.cpp(107)::addNewSceneInstanceToDataset : Dataset : 'default' : Lighting Layout Attributes 'no_lights' specified in Scene Attributes but does not exist in dataset, so creating default.
[20:16:49:350669]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `/Users/xinghebai/Desktop/Master Project/Sim Cross-view Dataset/replica_cad_baked_lighting/stages_uncompressed/Baked_sc3_staging_00.scn` does not exist.  Aborting load.
[20:16:49:350679]:[Warning]:[Sce

Renderer: Apple M2 Pro by Apple
OpenGL version: 4.1 Metal - 89.4
Simulator created for scene: /Users/xinghebai/Desktop/Master Project/Sim Cross-view Dataset/replica_cad_baked_lighting/stages_uncompressed/Baked_sc3_staging_00.glb
NavMesh not loaded, re-computing...
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_ES2_compatibility
    GL_ARB_separate_shader_objects
    GL_ARB_texture_storage
    GL_EXT_texture_filter_anisotropic
    GL_EXT_debug_label
    GL_EXT_debug_marker
Using driver workarounds:
    no-layout-qualifiers-on-old-glsl
    apple-buffer-texture-unbind-on-buffer-modify
Sampling 600 random camera poses with 4 yaw angles each...
Generated 2400 camera poses.
Starting frame rendering...


Rendering Baked_sc3_staging_00.glb: 100%|██| 2400/2400 [00:16<00:00, 145.07it/s]

Metadata saved to sim_cross_view_dataset/image_data.xlsx
Finished. Saved 519 sequential images (skipped 1881 bad frames).
Image generation complete.
